In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib as plt
from scipy.stats import chi2_contingency
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [11]:
#db connection

base_parquet_dir = "../data/interim/jobposts_base.parquet"

con = duckdb.connect()
con.read_parquet(base_parquet_dir).create_view("jobposts");

In [12]:
jobposts = con.sql(""" 
    select *
    from jobposts
""").df()

display(jobposts.head())

,RowNumber,Jobpost_RawTitle,Jobpost_PreferredGender,Jobpost_ProvinceFa,Jobpost_ProvinceEn,Jobpost_WorkTypeFa,Jobpost_WorkTypeEn,Jobpost_SalaryCanBeShown,Jobpost_MinSalary,Jobpost_MaxSalary,...,Company_ActivityTypeFa,Company_ActivityTypeEn,Company_IndustryFa,Company_IndustryEn,date,year,month,jalali_year,jalali_month,jalali_season
0,1,مهندس مکانیک,تفاوتی ندارد,تهران,Tehran,پاره وقت,Part Time,False,NaN,NaN,...,شرکت ایرانی دارای مشتریان داخلی,Iranian company dealing only with Iranian enti...,"ساختمان / مصالح و تجهیزات ساختمانی,تولیدی / صنعتی","Construction / Building Materials & Equipment,...",2022-04-01,2022,4,1401,1,Spring
1,2,طراح و گرافیست,تفاوتی ندارد,تهران,Tehran,تمام وقت,Full Time,False,NaN,NaN,...,شرکت ایرانی دارای مشتریان داخلی و خارجی,Iranian company dealing with Iranian and forei...,طراحی / گرافیک / هنر,Design / Arts,2023-03-01,2023,3,1401,12,Winter
2,3,برنامه نویس full-stack,تفاوتی ندارد,خراسان رضوی,Khorasan Razavi,تمام وقت,Full Time,False,NaN,NaN,...,شرکت ایرانی دارای مشتریان داخلی و خارجی,Iranian company dealing with Iranian and forei...,"اینترنت / تجارت الکترونیک / خدمات آنلاین,شرکت ...",Internet Provider / E-commerce / Online Servic...,2022-04-01,2022,4,1401,1,Spring
3,4,بازاریاب - خانم,فقط خانم,البرز,Alborz,تمام وقت,Full Time,False,NaN,NaN,...,شرکت ایرانی دارای مشتریان داخلی,Iranian company dealing only with Iranian enti...,"اینترنت / تجارت الکترونیک / خدمات آنلاین,خرده ...",Internet Provider / E-commerce / Online Servic...,2023-12-01,2023,12,1402,9,Autumn
4,5,برنامه نویس React Native,تفاوتی ندارد,اصفهان,Isfahan,پاره وقت,Part Time,False,NaN,NaN,...,شرکت ایرانی دارای مشتریان داخلی و خارجی,Iranian company dealing with Iranian and forei...,فناوری اطلاعات / نرم افزار و سخت افزار,IT / Software / Hardware,2022-07-01,2022,7,1401,4,Summer


### missing values

In [13]:

missing_overview = pd.DataFrame({
    "missing_count" : jobposts.isna().sum(),
    "missing_percent" : jobposts.isna().mean() * 100
})

missing_overview = missing_overview[missing_overview["missing_count"] > 0].sort_values(by="missing_percent",ascending=False)

display(missing_overview)

,missing_count,missing_percent
Jobpost_HasDisabilitySupport,346909,95.085764
Jobpost_SecondaryJobCategories,283525,77.712574
Jobpost_MaxSalary,244030,66.887221
Jobpost_MinSalary,244030,66.887221
Jobpost_LanguageSkills,235240,64.477933
Jobpost_AcademicFields,193088,52.924312
Jobpost_SoftwareSkills,168264,46.120196
Jobpost_RequiredMinAge,96363,26.412545
Jobpost_BenefitEn,90938,24.925583
Jobpost_BenefitFa,90938,24.925583


In [ ]:
#checking hidden missing values(empty string ,empty list and other placeholders)

jobposts_clean = jobposts.copy()

placeholders = {"", "[]", "{}", "null", "none", "nan", "-", "--", "n/a", "na"}

text_cols = jobposts_clean.select_dtypes(include=["object", "string"]).columns
report = {}
for col in text_cols:
    s = jobposts_clean[col]
    key = s.astype("string").str.strip().str.lower().str.replace(r"\s+", "", regex=True)
    mask = key.isin(placeholders) & s.notna()   # hidden placeholders only, not real NaN
    report[col] = int(mask.sum())
    jobposts_clean.loc[mask, col] = np.nan

placeholder_report = (pd.Series(report, name="placeholders_found")
                        .loc[lambda x: x > 0]
                        .sort_values(ascending=False))
display(placeholder_report)

Jobpost_CompanyWorkTimesDescription    204
Name: placeholders_found, dtype: int64

In [26]:
#compare to before checking placeholders

before = jobposts.isna().sum()
after = jobposts_clean.isna().sum()

comparison = pd.DataFrame({
    "before": before,
    "after": after,
    "new_missing": after - before,
})
comparison["after_percent"] = (comparison["after"] / len(jobposts_clean) * 100).round(2)

print(comparison[comparison["new_missing"] > 0])

                                     before  after  new_missing  after_percent
Jobpost_CompanyWorkTimesDescription      57    261          204           0.07


#### Missing value types 

**Not applicable (missing = the job doesn't need it)**
- `Jobpost_HasDisabilitySupport` (95.1%): flag likely only set when support is offered. Treat as "not offered", do not impute.
- `Jobpost_SecondaryJobCategories` (77.7%): most ads have one category only.
- `Jobpost_SoftwareSkills`, `Jobpost_LanguageSkills`, `Jobpost_AcademicFields` (46-65%): no requirement listed. In feature engineering, encode as 0 skills / no requirement.

**Optional by choice (advertiser could fill it, but didn't)**
- `Jobpost_MinSalary`, `Jobpost_MaxSalary` (66.9%)
- `Jobpost_RequiredMinAge`, `Jobpost_RequiredMaxAge` (24-26%): no age limit stated.
- `Jobpost_BenefitEn/Fa` (24.9%): no benefits listed.

**Truly missing (incomplete company profile or entry error)**
- `Company_ActivityType*`, `Company_CompanyOwnershipTypes*` (~24%): left as NaN, may become an "Unknown" category in models.
- `Company_City*` (0.05%), `Jobpost_CompanyWorkTimesDescription` (0.07%): negligible, left as NaN.

**Placeholder check:** only `Jobpost_CompanyWorkTimesDescription` had hidden placeholders (204 values converted to NaN). No other text column was affected. Numeric placeholders were checked.

### Salary Values

In [32]:
#jobposts salary can be shown
print("jobposts salary can be shown:")
display(jobposts_clean["Jobpost_SalaryCanBeShown"].value_counts().to_frame())

crosstab = pd.crosstab(
    jobposts_clean["Jobpost_SalaryCanBeShown"],
    [jobposts_clean["Jobpost_MinSalary"].notna().rename("has_min"),
     jobposts_clean["Jobpost_MaxSalary"].notna().rename("has_max")],
    margins=True,
)
display(crosstab)


print(f"number of nulls in salary column: {jobposts_clean["Jobpost_MinSalary"].isna().sum()}")

jobposts salary can be shown:


,count
Jobpost_SalaryCanBeShown,
False,244030
True,120808


has_min,False,True,All
has_max,False,True,
Jobpost_SalaryCanBeShown,,,
False,244030,0,244030
True,0,120808,120808
All,244030,120808,364838


number of nulls in salary column: 244030


#### Salary flag vs salary values

**Counts**
- `Jobpost_SalaryCanBeShown = True`: 120,808 ads (33.1%)
- `Jobpost_SalaryCanBeShown = False`: 244,030 ads (66.9%)
- No missing values in the flag itself.

**Consistency check**
- `False` with salary values: **0**
- `True` with empty salary values: **0**
- Only min or only max present: **0**
- `MinSalary` and `MaxSalary` are always present or missing together (244,030 nulls each, equal to the `False` count).

**Conclusion**
The flag and the salary values match exactly, so there are no contradictory or partial cases to resolve. Hiding the salary means both values are empty.


In [39]:
jobposts_clean["salary_disclosed"] = (
    jobposts_clean["Jobpost_SalaryCanBeShown"]
    & jobposts_clean["Jobpost_MinSalary"].notna()
    & jobposts_clean["Jobpost_MaxSalary"].notna()
)

#distribution of salary range
sal = jobposts_clean[jobposts_clean["salary_disclosed"]].copy()
sal["range"] = sal["Jobpost_MaxSalary"] - sal["Jobpost_MinSalary"]

display(sal[["Jobpost_MinSalary", "Jobpost_MaxSalary", "range"]].describe(percentiles=[.01, .25, .5, .75, .99]).round(2))


,Jobpost_MinSalary,Jobpost_MaxSalary,range
count,120808.00,120808.00,120808.00
mean,11.78,15.05,3.27
std,6.17,8.31,2.22
min,2.00,3.00,1.00
1%,5.00,6.00,1.00
25%,8.00,10.00,2.00
50%,10.00,12.00,2.00
75%,15.00,20.00,5.00
99%,35.00,45.00,10.00
max,70.00,100.00,30.00


In [ ]:
#Most common values and strange values
print(sal["Jobpost_MinSalary"].value_counts().head())
print(sal["Jobpost_MaxSalary"].value_counts().head())
print(sal["range"].value_counts().head())

#check Min > max count
print(f"\nnumber of ads with min salary > max salary :{(sal["Jobpost_MinSalary"] > sal["Jobpost_MaxSalary"]).sum()}")

Jobpost_MinSalary
12.0    25683
8.0     22980
10.0    19351
15.0    18713
6.0     13857
Name: count, dtype: int64
Jobpost_MaxSalary
15.0    25683
10.0    22979
12.0    19346
20.0    18715
8.0     13857
Name: count, dtype: int64
range
2.0     56186
5.0     26006
3.0     25686
1.0      6432
10.0     5813
Name: count, dtype: int64

number od ads with min salary > max salary :0


#### Salary values

**Distribution (120,808 disclosed ads)**
- Min salary: median 10, mean 11.8, range 2-70.
- Max salary: median 12, mean 15.1, range 3-100.
- Mean is above median and the maximum is far above the 99th percentile (min: 70 vs 35, max: 100 vs 45), so both are **right-skewed with a few extreme high values**. Outliers will be flagged (not deleted) in the EDA step.
- Salary range (max - min): median 2, mean 3.3, from 1 to 30.

**Most common values**
- Min: 12, 8, 10, 15, 6. Max: 15, 10, 12, 20, 8.
- All are round numbers. The counts match across the two columns (12 and 15 both ~25,683; 8 and 10 ~22,980; 10 and 12 ~19,350; 15 and 20 ~18,714; 6 and 8 ~13,857), which suggests advertisers pick from **fixed salary brackets** (12-15, 8-10, 10-12, 15-20, 6-8) rather than typing free values. This is an inference from the counts, not confirmed by the source.
- The most common ranges are 2 (46.5%), 5 (21.5%), 3 (21.3%), 1 (5.3%) and 10 (4.8%).

**Consistency checks**
- Min > max: **0 ads**, so no fixing is needed.
- Min = max (range 0): **0 ads**. The smallest range is 1, so every disclosed salary is a range, never a single fixed value.

**Implication**
Salaries are discrete bracket values, so `avg_salary` (the midpoint) will take a limited set of values. Later models and tests should be read with this in mind.

**Unit assumption**
Salaries are assumed to be **millions of Toman per month**. A median of 10-12 million is plausible for this period, but the unit is not yet verified against the source. Verify it before any money analysis (Income_Per_Hour, inflation comparison).

### Test Whether Salary Hiding Is Random?

In [51]:
#salary-disclosure rate by province, job category, industry, company size, company age, ownership type, employment type
cols = [
    "Jobpost_ProvinceEn",
    "Jobpost_MainJobCategory",
    "Jobpost_IndustryEn",
    "Company_SizeEn",
    "Company_AgeFromEstablishmentYear",
    "Company_CompanyOwnershipTypesEn",
    "Jobpost_WorkTypeEn",
]

# keep missing categories as their own group
for c in cols:
    jobposts_clean[c] = jobposts_clean[c].astype("object").fillna("Unknown")

def disclosure_rate(col):
    return (jobposts_clean.groupby(col)["salary_disclosed"]
              .agg(ads="count", disclosed="sum", rate="mean")
              .assign(rate=lambda x: (x["rate"] * 100).round(1))
              .sort_values("rate", ascending=False))

rates = {c: disclosure_rate(c) for c in cols}

for c, t in rates.items():
    print(f"\n--- {c} ---")
    print(t)


--- Jobpost_ProvinceEn ---
                            ads  disclosed  rate
Jobpost_ProvinceEn                              
Khorasan Shomali            593        280  47.2
Ilam                        704        329  46.7
Kohgilooye & Boyerahmad     481        221  45.9
Ardabil                    1135        494  43.5
Sistan & Baluchestan       1364        591  43.3
Khorasan Jonoobi            775        335  43.2
Azarbayjan Gharbi          2049        880  42.9
Lorestan                   1065        448  42.1
Hamedan                    2242        903  40.3
Azarbayjan Sharghi         5341       2116  39.6
Abroad                      329        130  39.5
Kerman                     3743       1475  39.4
Kermanshah                 1537        595  38.7
Hormozgan                  3017       1157  38.3
Booshehr                   1678        641  38.2
Kodestan                   1001        373  37.3
Mazandaran                 7612       2839  37.3
Yazd                       3897       144

In [50]:
#get chi-square test with cramer's v to find relationship between salary disclosure and each of these variables
from scipy.stats import chi2_contingency

def chi_cramers(col):
    ct = pd.crosstab(jobposts_clean[col], jobposts_clean["salary_disclosed"])
    chi2, p, dof, _ = chi2_contingency(ct)
    n = ct.values.sum()
    r, k = ct.shape

    # bias-corrected Cramér's V
    phi2corr = max(0, chi2 / n - (k - 1) * (r - 1) / (n - 1))
    rcorr = r - (r - 1) ** 2 / (n - 1)
    kcorr = k - (k - 1) ** 2 / (n - 1)
    v = np.sqrt(phi2corr / min(kcorr - 1, rcorr - 1))

    return {"variable": col, "chi2": round(chi2, 1), "dof": dof,
            "p_value": p, "cramers_v": round(v, 3)}

chi_results = (pd.DataFrame([chi_cramers(c) for c in cols])
                 .sort_values("cramers_v", ascending=False)
                 .reset_index(drop=True))

chi_results["strength"] = pd.cut(chi_results["cramers_v"],
                                 [-0.01, 0.1, 0.3, 1],
                                 labels=["weak", "moderate", "strong"])
display(chi_results)

,variable,chi2,dof,p_value,cramers_v,strength
0,Company_SizeEn,12876.3,6,0.000000e+00,0.188,moderate
1,Jobpost_MainJobCategory,10969.0,76,0.000000e+00,0.173,moderate
2,Jobpost_IndustryEn,7269.8,43,0.000000e+00,0.141,moderate
3,Company_AgeFromEstablishmentYear,2940.1,4,0.000000e+00,0.090,weak
4,Company_CompanyOwnershipTypesEn,2646.9,3,0.000000e+00,0.085,weak
5,Jobpost_WorkTypeEn,1088.3,3,1.249146e-235,0.055,weak
6,Jobpost_ProvinceEn,1096.9,31,1.037724e-210,0.054,weak


#### Chi-square and Cramér's V 

| Variable | Cramér's V (bias-corrected) | Strength |
|---|---|---|
| Company size | 0.188 | moderate |
| Main job category | 0.173 | moderate |
| Industry | 0.141 | moderate |
| Company age | 0.090 | weak |
| Ownership type | 0.085 | weak |
| Employment type | 0.055 | weak |
| Province | 0.054 | weak |

**Reading the results**
- All p-values are effectively 0 (smallest reported as < 1e-300). With 365K rows this only says the differences are not due to chance, not that they are large.
- No variable reaches a strong relationship (V > 0.3). The three largest are moderate: company size, job category and industry.
- **Company size** shows a clear gradient: 43.5% disclosure for 11-50 employees, down to 16.6% for 1001-5000.
- **Sector** matters: banking (12.1%) and telecom (13.6%) hide most, healthcare (53.5%) and real estate (52.7%) hide least.
- **Province** is the weakest (V = 0.054), so where the job is located matters little for hiding.
- Ownership V is partly driven by the "Unknown" group (24% of ads, 40.0% disclosure), which was kept as its own category.

**Conclusion so far**
Salary hiding is **not random**: it is associated with company size, job category and industry. But no single variable explains it. These variables overlap (large companies cluster in banking, telecom and manufacturing), so the logistic regression is needed to see which effects remain when the others are held fixed. These are associations, not causes.

In [55]:
#the logistic regression

import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

m = jobposts_clean.copy()
m["salary_hidden"] = (~m["salary_disclosed"]).astype(int)

#group rare categories (< 500 ads) into "Other"
def group_rare(s, min_n=500):
    return s.where(s.map(s.value_counts()) >= min_n, "Other")

for c in cols:                      
    m[c] = group_rare(m[c].astype(str))

#one-hot encode, dropping the most frequent level of each variable as the reference
X = pd.get_dummies(m[cols], dtype=float)
refs = {c: m[c].value_counts().idxmax() for c in cols}
X = X.drop(columns=[f"{c}_{r}" for c, r in refs.items()])
y = m["salary_hidden"]
print("refrence groups:")
print(refs)                       

# 3. train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# 4. fit and get odds ratios
logit = sm.Logit(y_train, sm.add_constant(X_train)).fit(disp=0)

or_table = pd.DataFrame({
    "odds_ratio": np.exp(logit.params),
    "ci_low": np.exp(logit.conf_int()[0]),
    "ci_high": np.exp(logit.conf_int()[1]),
}).drop("const").round(3)

# 5. AUC on the held-out test set
auc = roc_auc_score(y_test, logit.predict(sm.add_constant(X_test)))
print("\nAUC:", round(auc, 3))

# strongest effects in either direction
or_table["strength"] = np.abs(np.log(or_table["odds_ratio"]))
print("\nstrongest effects in either direction:")
display(or_table.sort_values("strength", ascending=False).head(15))

refrence groups:
{'Jobpost_ProvinceEn': 'Tehran', 'Jobpost_MainJobCategory': 'مالی و حسابداری', 'Jobpost_IndustryEn': 'Manufacturing & Production', 'Company_SizeEn': '11 - 50 employees', 'Company_AgeFromEstablishmentYear': 'more than 20 years', 'Company_CompanyOwnershipTypesEn': 'Privately held', 'Jobpost_WorkTypeEn': 'Full Time'}

AUC: 0.661

strongest effects in either direction:


,odds_ratio,ci_low,ci_high,strength
Company_SizeEn_1001 - 5000 employees,3.394,3.266,3.526,1.222009
Jobpost_MainJobCategory_پژوهش,3.195,2.603,3.922,1.161587
Company_SizeEn_More than 5001 employees,2.442,2.315,2.575,0.892817
Jobpost_IndustryEn_Banking,2.262,2.021,2.532,0.816249
Company_SizeEn_501 - 1000 employees,2.244,2.173,2.318,0.808260
Jobpost_MainJobCategory_علوم داده / هوش مصنوعی,2.209,1.789,2.728,0.792540
Jobpost_MainJobCategory_تست نرم افزار,2.207,1.744,2.794,0.791634
Jobpost_IndustryEn_Telecom,2.122,1.914,2.353,0.752359
Jobpost_MainJobCategory_مدیر محصول / مالک محصول,2.112,1.755,2.543,0.747635
Jobpost_MainJobCategory_مدیریت بیمه,0.476,0.432,0.525,0.742337


In [56]:
#ranking which variables matter most
def fit_auc(columns):
    lr = LogisticRegression(max_iter=1000)
    lr.fit(X_train[columns], y_train)
    return roc_auc_score(y_test, lr.predict_proba(X_test[columns])[:, 1])

all_cols = list(X.columns)
full_auc = fit_auc(all_cols)

drops = {}
for c in cols:
    keep = [k for k in all_cols if not k.startswith(c + "_")]
    drops[c] = full_auc - fit_auc(keep)

importance = pd.Series(drops, name="auc_drop").sort_values(ascending=False).round(4)
print(importance)

Company_SizeEn                      0.0217
Jobpost_MainJobCategory             0.0177
Jobpost_IndustryEn                  0.0081
Company_AgeFromEstablishmentYear    0.0035
Jobpost_ProvinceEn                  0.0024
Company_CompanyOwnershipTypesEn     0.0010
Jobpost_WorkTypeEn                  0.0003
Name: auc_drop, dtype: float64
